<a href="https://colab.research.google.com/github/wongjustin2/Econ_3916_PS2/blob/main/Problem%20Set%202/Econ_3916_PS2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS2 — The Survey That Wasn't Random

**ECON 3916 · Applied Data Analytics** · Topics 5-8 · 100 points · your role: Data Analyst, Growth team

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is data analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. You write the simulations, the bootstrap loop and the tests yourself. AI is authorised and required in Phase 4. If you cannot explain a line you submitted, you cannot submit it.

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Simulate the Population You Cannot See (30 pts)

### Step 1.1: Build a world with a known answer (15 pts)

In [2]:
# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_USERS = 50_000
BASE_BEFORE, BASE_AFTER = 3.432, 3.532      # true means 3.30 and 3.40

def simulate_population(base, n=N_USERS, rng=rng):
    """Return a DataFrame with engagement_minutes, engagement_z, had_support_ticket,
    latent s and the reported 1–5 satisfaction score."""
    engagement_minutes = rng.lognormal(mean=3.5, sigma=0.6, size=n)
    engagement_z = (engagement_minutes - engagement_minutes.mean()) / engagement_minutes.std()
    had_support_ticket = rng.binomial(1, 0.22, size=n)
    s = base + 0.35 * engagement_z - 0.60 * had_support_ticket + rng.normal(0, 0.75, size=n)
    score = s.round().clip(1, 5)
    return pd.DataFrame({"engagement_minutes": engagement_minutes,
                         "engagement_z": engagement_z,
                         "had_support_ticket": had_support_ticket,
                         "s": s,
                         "score": score})

pre = simulate_population(BASE_BEFORE)
post = simulate_population(BASE_AFTER)

true_pre = pre["score"].mean()
true_post = post["score"].mean()
print(f"True mean before:  latent s {pre['s'].mean():.3f}   reported score {true_pre:.3f}")
print(f"True mean after:   latent s {post['s'].mean():.3f}   reported score {true_post:.3f}")
print(f"True improvement:  latent s {post['s'].mean() - pre['s'].mean():.3f}   reported score {true_post - true_pre:.3f}")

True mean before:  latent s 3.299   reported score 3.290
True mean after:   latent s 3.399   reported score 3.386
True improvement:  latent s 0.100   reported score 0.096


*Your answer here.*

I recorded the true mean satisfaction as 3.30 before and 3.40 after in latent s (3.29 and 3.39 in the reported 1 to 5 score) so the true improvement is 0.10 points.

### Step 1.2: Impose the response mechanism (15 pts)

In [3]:
def respond(df, a, b, rng=rng):
    """Who answers the survey: P(respond) = logistic(a + b*(s - 3.3) + 0.30*engagement_z
    - 0.25*had_support_ticket); a user answers when a uniform draw is below it."""
    z = a + b * (df["s"] - 3.3) + 0.30 * df["engagement_z"] - 0.25 * df["had_support_ticket"]
    p_respond = 1 / (1 + np.exp(-z))
    return rng.random(len(df)) < p_respond

resp_pre = respond(pre, a=-1.0, b=0.45)
resp_post = respond(post, a=-2.75, b=2.00)

obs_pre = pre.loc[resp_pre, "score"]
obs_post = post.loc[resp_post, "score"]

print(f"Observed mean before: {obs_pre.mean():.3f}   response rate {resp_pre.mean():.1%}   respondents {resp_pre.sum():,}")
print(f"Observed mean after:  {obs_post.mean():.3f}   response rate {resp_post.mean():.1%}   respondents {resp_post.sum():,}")
print(f"Apparent improvement: {obs_post.mean() - obs_pre.mean():.3f}   true improvement {true_post - true_pre:.3f}")

Observed mean before: 3.600   response rate 27.5%   respondents 13,768
Observed mean after:  4.304   response rate 15.2%   respondents 7,597
Apparent improvement: 0.704   true improvement 0.096


> **🛑 Verification checkpoint.** You should get an observed pre-redesign mean of
> about **3.60** (accept 3.55 to 3.65) and an observed post-redesign mean of
> about **4.31** (accept 4.25 to 4.35) — an apparent gap of roughly **0.71**
> points out of a true improvement of **0.10**. Response rates should fall from
> roughly 27% to roughly 15%. If your numbers are outside those bands, your
> response function or your `base` values are wrong. **Do not proceed until it
> matches** — everything after this depends on reproducing the PM's slide.

*Your answer here.*

This is Chapter 6's nonresponse bias and it's MNAR because the chance of answering depends on satisfaction itself so the users who answer are happier than the ones who don't, and a larger survey wouldn't fix it because more answers with the same bias will only lead to the same wrong answer. The response rate fell from 27.5% to 15.2% so the survey that produced the better number heard from fewer people.

## Phase 2: Quantify What You Actually Know (35 pts)

### Step 2.1: Uncertainty without a formula (20 pts)

In [4]:
def bootstrap_by_hand(x, stat, B=2000, rng=rng):
    """Your own resampling loop; returns the B bootstrap statistics."""
    return np.array([stat(rng.choice(x, size=len(x), replace=True)) for _ in range(B)])

x = obs_post.values      # post-redesign survey scores, the 4.3 on the slide
boot_mean = bootstrap_by_hand(x, np.mean)
boot_median = bootstrap_by_hand(x, np.median)
print(f"Mean    {np.mean(x):.3f}   by hand percentile [{np.percentile(boot_mean, 2.5):.3f}, {np.percentile(boot_mean, 97.5):.3f}]")
print(f"Median  {np.median(x):.3f}   by hand percentile [{np.percentile(boot_median, 2.5):.3f}, {np.percentile(boot_median, 97.5):.3f}]")

for stat, name in [(np.mean, "Mean"), (np.median, "Median")]:
    for method in ["percentile", "BCa"]:
        ci = stats.bootstrap((x,), stat, n_resamples=2000, confidence_level=0.95,
                             method=method, random_state=rng).confidence_interval
        print(f"{name:7s} scipy {method:10s} [{ci.low:.3f}, {ci.high:.3f}]")

Mean    4.304   by hand percentile [4.289, 4.319]
Median  4.000   by hand percentile [4.000, 4.000]
Mean    scipy percentile [4.290, 4.319]
Mean    scipy BCa        [4.288, 4.319]
Median  scipy percentile [4.000, 4.000]
Median  scipy BCa        [nan, nan]


/usr/local/lib/python3.13/dist-packages/scipy/stats/_resampling.py:147: RuntimeWarning: invalid value encountered in scalar divide
  a_hat = 1/6 * sum(nums) / sum(dens)**(3/2)
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:352: DegenerateDataWarning: The BCa confidence interval cannot be calculated. This problem is known to occur when the distribution is degenerate or the statistic is np.min.
  return fun(*args, **kwargs)


*Your answer here.*

My loop and stats.bootstrap agree with 95% percentile intervals for the mean of [4.289, 4.319] and [4.290, 4.319]. The BCa interval of [4.288, 4.319] differs from the percentile interval only at the lower end by 0.002 because the scores are skewed toward 4 and 5. The median is 4 in every resample so both percentile intervals are [4.000, 4.000] and BCa can't be computed (nan).

### Step 2.2: Coverage is a claim you can check (15 pts)

In [5]:
R = 1000
hits = {"Real survey, percentile": 0, "Real survey, BCa": 0,
        "Random survey, percentile": 0, "Random survey, BCa": 0}

for r in range(R):
    invited = post.loc[rng.choice(N_USERS, size=2000, replace=False)]
    real = invited.loc[respond(invited, a=-2.75, b=2.00), "score"].values
    random_ = rng.choice(post["score"].values, size=len(real), replace=False)
    for survey, sample in [("Real survey", real), ("Random survey", random_)]:
        for method in ["percentile", "BCa"]:
            ci = stats.bootstrap((sample,), np.mean, n_resamples=999, confidence_level=0.95,
                                 method=method, random_state=rng).confidence_interval
            if ci.low <= true_post <= ci.high:
                hits[f"{survey}, {method}"] += 1

for name, h in hits.items():
    coverage = h / R
    mc_se = np.sqrt(coverage * (1 - coverage) / R)
    print(f"{name:26s} coverage {coverage:.1%}   Monte Carlo SE {mc_se:.1%}")

Real survey, percentile    coverage 0.0%   Monte Carlo SE 0.0%
Real survey, BCa           coverage 0.0%   Monte Carlo SE 0.0%
Random survey, percentile  coverage 94.5%   Monte Carlo SE 0.7%
Random survey, BCa         coverage 94.7%   Monte Carlo SE 0.7%


*Your answer here.*

Across 1,000 repeated surveys the percentile and BCa intervals from the real survey contained the truth (3.39) 0.0% of the time (Monte Carlo SE 0.0%) and a random survey of the same size contained the truth 94.5% and 94.7% of the time (Monte Carlo SE 0.7% each) so the 0.2 percentage point gap between percentile and BCa is smaller than the 0.7% Monte Carlo SE and isn't a finding.

## Phase 3: Test Something Worth Testing (20 pts)

### Step 3.1: The test the PM wants

In [6]:
t_stat, p_val = stats.ttest_ind(obs_post, obs_pre, equal_var=False)
diff = obs_post.mean() - obs_pre.mean()
n_post, n_pre = len(obs_post), len(obs_pre)
sd_pooled = np.sqrt(((n_post - 1) * np.std(obs_post, ddof=1)**2
                     + (n_pre - 1) * np.std(obs_pre, ddof=1)**2) / (n_post + n_pre - 2))
cohens_d = diff / sd_pooled
print(f"Observed difference {diff:.3f} points   t = {t_stat:.2f}   p = {p_val:.4f}   Cohen's d = {cohens_d:.3f}")

Observed difference 0.704 points   t = 65.17   p = 0.0000   Cohen's d = 0.864


*Your answer here.*

The two-sample test is significant (t = 65.17 and p < 0.0001) with a large effect size of Cohen's d = 0.864 but statistical significance isn't the issue here because the test only shows the 0.704 point difference between the people who answered isn't due to chance and can't tell that most of this difference comes from who chose to answer.

### Step 3.2: Power, before the fact

In [7]:
from statsmodels.stats.power import TTestIndPower

power_analysis = TTestIndPower()
sd_plan = np.std(obs_pre, ddof=1)     # spread of the pre-redesign survey, known before the follow-up
n_80 = power_analysis.solve_power(effect_size=0.10 / sd_plan, alpha=0.05, power=0.80,
                                  ratio=1.0, alternative="two-sided")
d_detect = power_analysis.solve_power(nobs1=n_post, alpha=0.05, power=0.80,
                                      ratio=n_pre / n_post, alternative="two-sided")
print(f"Respondents per group for 80% power at 0.10 points (d = {0.10 / sd_plan:.3f}): {n_80:,.0f}")
print(f"Detectable at {n_post:,} and {n_pre:,} respondents: d = {d_detect:.3f} = {d_detect * sd_plan:.3f} points")

for n in [1_000, 10_000, 100_000]:
    est = []
    for rep in range(200):
        pre_sim = simulate_population(BASE_BEFORE, n=n)
        post_sim = simulate_population(BASE_AFTER, n=n)
        pre_obs = pre_sim.loc[respond(pre_sim, a=-1.0, b=0.45), "score"]
        post_obs = post_sim.loc[respond(post_sim, a=-2.75, b=2.00), "score"]
        est.append(post_obs.mean() - pre_obs.mean())
    est = np.array(est)
    print(f"Users per period {n:>7,}   mean estimate {est.mean():.3f}   SD {est.std(ddof=1):.3f}   error vs true 0.10 {est.mean() - 0.10:.3f}")

Respondents per group for 80% power at 0.10 points (d = 0.113): 1,224
Detectable at 7,597 and 13,768 respondents: d = 0.040 = 0.035 points
Users per period   1,000   mean estimate 0.715   SD 0.073   error vs true 0.10 0.615
Users per period  10,000   mean estimate 0.710   SD 0.023   error vs true 0.10 0.610
Users per period 100,000   mean estimate 0.711   SD 0.008   error vs true 0.10 0.611


*Your answer here.*

For a true improvement of 0.1 points a sample size of about 1,224 respondents per group gives 80% power and at the sample size they can actually get (7,597 and 13,768 respondents) the detectable effect size is d = 0.040 or 0.035 points, and no a larger sample wouldn't make the estimate better because in my simulation going from 1,000 to 100,000 users per period shrinks the SD of the estimate from 0.073 to 0.008 but the estimate stays at about 0.71 points or 0.61 above the true 0.10.

## Phase 4: AI Expansion — The Co-Pilot Era (15 pts)

**Status: AI usage AUTHORISED and REQUIRED.** You have built the simulation, the bootstrap and the power analysis by hand. Now use the **P.R.I.M.E. framework** to go further than you could in the time.

### Task 4.1: The weighting attempt (10 pts)

In [8]:

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

# Use the existing simulated population and response indicators.
# engagement_z is excluded because it is a standardized
# version of engagement_minutes, not a separate covariate.

X = post[["engagement_minutes", "had_support_ticket"]].copy()
X["had_support_ticket"] = X["had_support_ticket"].astype(int)

y = np.asarray(resp_post, dtype=bool)

def ipw_mean(X, y, score):
    # Fit logistic regression to estimate response probabilities.
    model = LogisticRegression(max_iter=1000)
    model.fit(X, y.astype(int))

    # Predict response probabilities for all users.
    p_response = model.predict_proba(X)[:, 1]

    # Use satisfaction scores only for respondents.
    weights = 1.0 / p_response[y]

    # Normalized IPW (Hajek) weighted mean.
    return np.average(score[y], weights=weights)


# Calculate observed and weighted satisfaction means.
score = post["score"].to_numpy()

observed_mean = score[y].mean()
weighted_estimate = ipw_mean(X, y, score)

# Bootstrap: resample users and refit the response model.
bootstrap_estimates = []

for _ in range(1000):
    indices = rng.integers(0, len(post), size=len(post))

    X_boot = X.iloc[indices]
    y_boot = y[indices]
    score_boot = score[indices]

    # Both response classes are needed for logistic regression.
    if np.unique(y_boot).size < 2:
        continue

    estimate = ipw_mean(X_boot, y_boot, score_boot)
    bootstrap_estimates.append(estimate)

# Calculate 95% percentile bootstrap confidence interval.
ci_lower, ci_upper = np.percentile(
    bootstrap_estimates, [2.5, 97.5]
)

print(f"Observed survey mean: {observed_mean:.3f}")
print(f"IPW weighted estimate: {weighted_estimate:.3f}")
print(f"95% bootstrap CI: [{ci_lower:.3f}, {ci_upper:.3f}]")
print(f"True population mean: {true_post:.3f}")
print(f"Remaining bias: {weighted_estimate - true_post:.3f}")


Observed survey mean: 4.304
IPW weighted estimate: 4.143
95% bootstrap CI: [4.122, 4.164]
True population mean: 3.386
Remaining bias: 0.757


*Your answer here.*

I changed the [Iterate] block to give my variable names because version 1 said "Reuse my variable names" but never gave them so the AI made up its own names and the code didn't run on my data. The corrected mean is 4.143 (95% CI [4.122, 4.164]) against my known true mean of 3.386 so the correction didn't work and only moved the observed mean of 4.304 down by 0.161 points because response depends on satisfaction itself and the logistic model only uses engagement and support tickets. The logistic model uses engagement_minutes and had_support_ticket to predict each user's chance of answering and each respondent's weight is 1 divided by that chance so respondents who look unlikely to answer count more. np.average then takes the mean score with those weights. Yes I'd use AI for this step again because it wrote working weighting code fast but I could only trust the result because I knew the true mean.

### Task 4.2: The slide (5 pts)

*Your answer here.*

The survey shows that the users who answered after the redesign gave higher scores than the users who answered before it but it doesn't show that our users' satisfaction went up from 3.6 to 4.3 or that the redesign worked because after the redesign fewer users answered and most of them were satisfied users so the higher score mostly comes from who answered and a bigger survey wouldn't fix this. The users who answered after the redesign gave an average score of 4.30 (95% CI 4.29 to 4.32) but this number is only about the users who answered and not about all users. To fix this we should send the survey to a random group of users who finish onboarding and keep following up until almost all of them answer so who answers no longer depends on how satisfied they are.

### AI Transcript

Paste **both prompt versions exactly as you sent them** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

###*Model and date:* ChatGPT Oct 26

###*Version 1 prompt:*

[Prep] Act as a survey statistician who works on non-response
adjustment for product research.

[Request] I have a simulated user population with satisfaction scores
and two covariates, engagement_minutes and had_support_ticket, plus a
non-response mechanism where the probability of responding increases
with satisfaction. Write Python that implements inverse-probability
weighting to correct the observed survey mean, using a logistic model
of response on the covariates.

[Iterate] numpy, pandas, scikit-learn or statsmodels. Reuse my
variable names. Return the weighted estimate and its bootstrap
interval.

[Mechanism Check] Explain precisely which assumption makes this
correction valid, and what happens to it when response depends on
satisfaction ITSELF rather than only on the covariates.

[Evaluate] Given my mechanism, will this correction recover the true
mean? Answer yes or no and justify it.

###*Reply:*

####1. Python: Inverse-Probability Weighting (IPW)

The goal is to correct a survey mean when people with higher satisfaction scores are more likely to respond.

We'll use logistic regression to estimate each person's probability of responding based on two covariates:

- `engagement_minutes`
- `had_support_ticket`

Then we'll use inverse-probability weights to calculate the adjusted satisfaction mean and a 95% bootstrap confidence interval.

The important limitation is that IPW only removes non-response bias when the variables in the response model adequately explain why people respond.

Your mechanism depends directly on satisfaction, so this particular correction is not guaranteed to recover the true population mean.

####2. Complete Python code

This code assumes your simulated population is stored in a DataFrame called `df`, with columns `satisfaction`, `engagement_minutes`, `had_support_ticket`, and `responded` (1 = responded, 0 = did not respond).
```
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

Full simulated population.
Required columns:
satisfaction, engagement_minutes, had_support_ticket, responded

covariates = ["engagement_minutes", "had_support_ticket"]

def calculate_ipw(data):
    # Fit the response model using all population members.
    X = data[covariates]
    y = data["responded"]

    model = LogisticRegression(max_iter=1000)
    model.fit(X, y)

    # Estimated probability of responding.
    data = data.copy()
    data["response_probability"] = model.predict_proba(X)[:, 1]

    # Only respondents have observed satisfaction in a real survey.
    observed = data.loc[data["responded"] == 1].copy()

    # Inverse-probability weights.
    observed["weight"] = (
        1 / observed["response_probability"]
    )

    # Normalized IPW (Hájek) estimate.
    weighted_mean = np.average(
        observed["satisfaction"],
        weights=observed["weight"]
    )

    return weighted_mean


# Unadjusted survey mean.
observed_mean = df.loc[
    df["responded"] == 1, "satisfaction"
].mean()

# IPW-adjusted estimate.
weighted_estimate = calculate_ipw(df)

# True mean: available because this is simulated data.
true_mean = df["satisfaction"].mean()

# Bootstrap: resample population records and refit the model.
rng = np.random.default_rng(42)
n_bootstrap = 500
bootstrap_estimates = []

for _ in range(n_bootstrap):
    indices = rng.choice(
        len(df), size=len(df), replace=True
    )

    sample = df.iloc[indices].copy()

    # Logistic regression requires both response classes.
    if sample["responded"].nunique() < 2:
        continue

    estimate = calculate_ipw(sample)
    bootstrap_estimates.append(estimate)

# Percentile 95% confidence interval.
lower, upper = np.percentile(
    bootstrap_estimates, [2.5, 97.5]
)

print(f"Observed survey mean: {observed_mean:.3f}")
print(f"IPW weighted estimate: {weighted_estimate:.3f}")
print(f"95% bootstrap CI: [{lower:.3f}, {upper:.3f}]")
print(f"True population mean: {true_mean:.3f}")
print(f"Remaining IPW bias: {weighted_estimate - true_mean:.3f}")
```

The code reports the weighted estimate, its bootstrap interval, and how far the estimate is from the true simulated mean. Actual numerical results require your `df` data.

Because the bootstrap resamples complete simulated population records, it reflects repeated sampling of those records. It does not automatically give a valid confidence interval under a misspecified response model.

####3. Mechanism check: When does IPW work?

IPW relies on three important conditions.

1\. Missing at random (MAR)

After accounting for the covariates, satisfaction must not independently influence whether somebody responds.

Mathematically:

\\[ R\perp Y\mid X \\]

Where:

- \\(R\\) = whether someone responds
- \\(Y\\) = satisfaction score
- \\(X\\) = engagement minutes and support ticket status

2\. Positivity

Every group of users must have a nonzero probability of responding:

\\[ 0\<P(R=1\mid X)<1 \\\]

Otherwise, we cannot reliably recover information about groups that never respond.

3\. Correct response model

The estimated logistic regression must adequately represent the actual response probabilities. Logistic regression does not guarantee this automatically.

The IPW formula is:

\\[ \hat{\mu}\_{IPW} = \frac{\sum\_{i:R_i=1}Y_i/\hat p_i} {\sum\_{i:R_i=1}1/\hat p_i} \\]

Here, \\(\hat p_i\\) represents the estimated response probability for individual \\(i\\).

####4. What happens when response depends directly on satisfaction?

Your simulation says that people with higher satisfaction scores are more likely to respond.

That introduces a problem.

Even among users with identical engagement minutes and support ticket histories, satisfied users can still be more likely to respond than dissatisfied users.

This is missing not at random (MNAR), relative to your observed covariates.

A model using only `engagement_minutes` and `had_support_ticket` cannot fully capture that difference.

Consequently:

- Lower-satisfaction users can remain underrepresented.
- The weighted mean can remain too high.
- The bootstrap interval can be narrow while the estimate is biased.
- Increasing the sample size does not necessarily eliminate the bias.

####5. Evaluation: Will IPW recover the true mean?

No — not reliably.

MAR violated

Because satisfaction directly affects the response probability, and satisfaction itself is excluded from the logistic response model, the missing-at-random assumption generally fails.

The correction might reduce bias, but it will not generally recover the true population mean. It could match by coincidence or in a special case where the covariates fully capture the relevant satisfaction effect.

For a simulated dataset, one option is to compare your estimated mean against the known true population mean, as the code does. You can also use the known response mechanism to study how much bias remains.

For real survey data, where satisfaction is unobserved for nonrespondents, you would typically need additional information or a sensitivity analysis to assess and address this bias.

Bottom line: IPW works when response differences are adequately explained by observed covariates. When response still depends directly on the missing satisfaction score, covariate-only IPW is generally insufficient.

###*Version 2 prompt:*

[Prep] Act as a survey statistician who works on non-response
adjustment for product research.

[Request] I have a simulated user population with satisfaction scores
and two covariates, engagement_minutes and had_support_ticket, plus a
non-response mechanism where the probability of responding increases
with satisfaction. Write Python that implements inverse-probability
weighting to correct the observed survey mean, using a logistic model
of response on the covariates.

[Iterate] numpy, pandas, scikit-learn or statsmodels. Reuse my
variable names: my post-redesign population is a DataFrame called post
with columns engagement_minutes, engagement_z, had_support_ticket and
score, resp_post is a boolean Series that is True for users who
answered, true_post is the true mean and rng = np.random.default_rng(42)
is my generator. Return the weighted estimate and its bootstrap
interval.

[Mechanism Check] Explain precisely which assumption makes this
correction valid, and what happens to it when response depends on
satisfaction ITSELF rather than only on the covariates.

[Evaluate] Given my mechanism, will this correction recover the true
mean? Answer yes or no and justify it.

###*Reply:*

#### 1. Python: Inverse-Probability Weighting (IPW)

This code uses your existing variables: `post`, `resp_post`, `true_post`, and `rng`.

It fits a logistic regression model to estimate response probabilities using `engagement_minutes` and `had_support_ticket`, then calculates the IPW-adjusted mean and a 95% bootstrap confidence interval.

```

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

# Use the existing simulated population and response indicators.
# engagement_z is excluded because it is a standardized
# version of engagement_minutes, not a separate covariate.

X = post[["engagement_minutes", "had_support_ticket"]].copy()
X["had_support_ticket"] = X["had_support_ticket"].astype(int)

y = np.asarray(resp_post, dtype=bool)

def ipw_mean(X, y, score):
    # Fit logistic regression to estimate response probabilities.
    model = LogisticRegression(max_iter=1000)
    model.fit(X, y.astype(int))

    # Predict response probabilities for all users.
    p_response = model.predict_proba(X)[:, 1]

    # Use satisfaction scores only for respondents.
    weights = 1.0 / p_response[y]

    # Normalized IPW (Hajek) weighted mean.
    return np.average(score[y], weights=weights)


# Calculate observed and weighted satisfaction means.
score = post["score"].to_numpy()

observed_mean = score[y].mean()
weighted_estimate = ipw_mean(X, y, score)

# Bootstrap: resample users and refit the response model.
bootstrap_estimates = []

for _ in range(1000):
    indices = rng.integers(0, len(post), size=len(post))

    X_boot = X.iloc[indices]
    y_boot = y[indices]
    score_boot = score[indices]

    # Both response classes are needed for logistic regression.
    if np.unique(y_boot).size < 2:
        continue

    estimate = ipw_mean(X_boot, y_boot, score_boot)
    bootstrap_estimates.append(estimate)

# Calculate 95% percentile bootstrap confidence interval.
ci_lower, ci_upper = np.percentile(
    bootstrap_estimates, [2.5, 97.5]
)

print(f"Observed survey mean: {observed_mean:.3f}")
print(f"IPW weighted estimate: {weighted_estimate:.3f}")
print(f"95% bootstrap CI: [{ci_lower:.3f}, {ci_upper:.3f}]")
print(f"True population mean: {true_post:.3f}")
print(f"Remaining bias: {weighted_estimate - true_post:.3f}")

```

#### What the code does

1. Logistic regression: Estimates how likely each user is to respond based on engagement and support ticket history.
2. Inverse-probability weighting: Gives more weight to respondents with a lower predicted response probability.
3. Weighted estimate: Calculates the adjusted average satisfaction score.
4. Bootstrap interval: Resamples users 1,000 times, refitting the model each time to estimate uncertainty.
5. Bias comparison: Compares the adjusted estimate with `true_post`, which is available because your data is simulated.

The bootstrap interval describes sampling uncertainty under this procedure. It does not correct bias caused by an invalid response model.

#### 2. Mechanism check: When does IPW work?

The central assumption is Missing at Random (MAR).

This means that after accounting for engagement minutes and support ticket history, whether someone responds must not depend on their satisfaction score.

Mathematically:

\\[ R\perp Y\mid X \\]

Where:

- \\(R\\) = response indicator (`resp_post`)
- \\(Y\\) = satisfaction score (`post["score"]`)
- \\(X\\) = engagement minutes and support ticket history

For IPW to consistently recover the mean, we also need positive response probabilities for all relevant covariate groups and correctly estimated response probabilities.

Under these conditions, the weighted estimator is:

\\[ \hat{\mu}\_{IPW}= \frac{\sum\_{i:R_i=1}Y_i/\hat p_i} {\sum\_{i:R_i=1}1/\hat p_i} \\]

Here, \\(\hat p_i\\) is the predicted probability that user \\(i\\) responds.

#### 3. What happens when response depends on satisfaction itself?

Your non-response mechanism makes people with higher satisfaction scores more likely to answer the survey.

This creates Missing Not at Random (MNAR) when satisfaction continues to influence response after conditioning on the two covariates.

For example, imagine two users have identical engagement minutes and support ticket histories, but different satisfaction scores.

The logistic model assigns them the same predicted response probability. However, your simulation may make the more satisfied user more likely to respond.

That difference cannot be corrected using only the two covariates.

As a result, the weighted estimate can remain biased upward, even if the logistic regression is fitted correctly to the observed covariates.

#### 4. Evaluation: Will this correction recover the true mean?

No, not reliably.

Because the response mechanism directly favors users with higher satisfaction scores, covariate-only IPW generally does not satisfy the MAR assumption.

The adjustment may reduce the difference between the observed survey mean and `true_post`, especially if engagement and support ticket history are related to satisfaction. However, it cannot generally eliminate the remaining bias.

Even with a very large sample, the adjusted estimate may converge to the wrong population mean.

Conclusion: IPW can correct non-response bias when response depends on adequately modeled observed covariates. In your simulation, response also depends directly on satisfaction, so the proposed logistic IPW model will generally not recover `true_post`. The code lets you measure the remaining bias, but it cannot guarantee its elimination.

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; both prompt versions are in it.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 2/Econ_3916_PS2.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.